### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [35]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [36]:
### Read all PDF's from directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #add source info to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata["source_path"] = str(pdf_file)
                doc.metadata['file_type'] = 'pdf'     

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all the documents in the PDF directory
all_pdf_documents = process_all_pdfs("../data")


Found 2 PDF files to process

Processing: Aman Agrahari 5SFS.pdf
 Loaded 1 pages

Processing: Aman Agrahari FS4S.pdf
 Loaded 1 pages

Total documents loaded: 2


In [37]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [38]:
chunks = split_documents(all_pdf_documents)
chunks

Split 2 documents into 9 chunks

Example chunk:
Content: Aman Agrahari
7234909407
G I T H U B | |
EDUCATION
EXPERIENCE
Web3Task  
Full Stack Developer Internship 
Sept 2024 - Aug 2028
April 2025 - Present
April 2026 - May 2026
aman8cse@gmail.com |
PRODUCTS ...
Metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-06-03T14:15:36+00:00', 'title': 'resume template.pdf', 'moddate': '2026-06-03T14:15:36+00:00', 'keywords': 'DAHELWz6xSo,BAGZGbj1AHg,0', 'author': 'Aman', 'trapped': '/False', 'source': '..\\data\\pdf\\Aman Agrahari 5SFS.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Aman Agrahari 5SFS.pdf', 'source_path': '..\\data\\pdf\\Aman Agrahari 5SFS.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-06-03T14:15:36+00:00', 'title': 'resume template.pdf', 'moddate': '2026-06-03T14:15:36+00:00', 'keywords': 'DAHELWz6xSo,BAGZGbj1AHg,0', 'author': 'Aman', 'trapped': '/False', 'source': '..\\data\\pdf\\Aman Agrahari 5SFS.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Aman Agrahari 5SFS.pdf', 'source_path': '..\\data\\pdf\\Aman Agrahari 5SFS.pdf', 'file_type': 'pdf'}, page_content='Aman Agrahari\n7234909407\nG I T H U B | |\nEDUCATION\nEXPERIENCE\nWeb3Task  \nFull Stack Developer Internship \nSept 2024 - Aug 2028\nApril 2025 - Present\nApril 2026 - May 2026\naman8cse@gmail.com |\nPRODUCTS & PROJECTS\nWatchroom - Real-Time Watch Party Platform\nLIC Calc - Insurance Analytics & Advisory Engine\nEngineered a real-time watch party platform enabling synchronized YouTube playback, live chat, reactions.\nBuilt room ownership, moderator controls, participant permissions, and session lifecycl